In [44]:
import pandas as pd
import numpy as np
import os
import glob

In [45]:
DATA_PATH = "../data/raw/nba_pbp"
all_files = sorted(glob.glob(os.path.join(DATA_PATH, "*.csv")))

dfs = []

for file in all_files:
    df_temp = pd.read_csv(file)

    season = os.path.basename(file).replace("pbp", "").replace(".csv", "")
    df_temp["season"] = int(season)

    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print(df.shape)
df.head()

(18255730, 16)


,gameid,period,clock,h_pts,a_pts,team,playerid,player,type,subtype,result,x,y,dist,desc,season
0,29600001,1,PT12M00.00S,0.0,0.0,NaN,0,NaN,period,start,NaN,0.0,0.0,0.0,Start of 1st Period (11:15 PM EST),1997
1,29600001,1,PT12M00.00S,0.0,0.0,BOS,442,P. Ellison,Jump Ball,NaN,NaN,0.0,0.0,0.0,Jump Ball Ellison vs. Longley: Tip to Harper,1997
2,29600001,1,PT11M39.00S,0.0,2.0,CHI,23,D. Rodman,Made Shot,Layup Shot,Made,0.0,0.0,0.0,Rodman Layup (2 PTS) (Longley 1 AST),1997
3,29600001,1,PT11M39.00S,0.0,0.0,BOS,442,P. Ellison,Foul,Shooting,NaN,0.0,0.0,0.0,Ellison S.FOUL (P1.T1),1997
4,29600001,1,PT11M39.00S,0.0,3.0,CHI,23,D. Rodman,Free Throw,Free Throw 1 of 1,NaN,0.0,0.0,0.0,Rodman Free Throw 1 of 1 (3 PTS),1997


In [46]:
df = df.sort_values(["gameid", "period", "clock"]).reset_index(drop=True)

In [47]:
df["h_pts"] = df.groupby("gameid")["h_pts"].ffill()
df["a_pts"] = df.groupby("gameid")["a_pts"].ffill()

In [48]:
df["score_diff"] = df["h_pts"] - df["a_pts"]

In [49]:
df["score_diff_squared"] = df["score_diff"] ** 2

In [50]:
def convert_clock(clock):
    clock = clock.replace("PT", "")

    minutes = clock.split("M")[0]
    seconds = clock.split("M")[1].replace("S", "")

    return int(minutes) * 60 + float(seconds)


df["quarter_seconds_elapsed"] = df["clock"].apply(convert_clock)

MemoryError: Unable to allocate 139. MiB for an array with shape (18255730,) and data type float64

In [ ]:
df["quarter_seconds_remaining"] = (
    720 - df["quarter_seconds_elapsed"]
)

In [ ]:
def convert_game_clock(row):

    quarter = row["period"]
    quarter_time = convert_clock(row["clock"])

    # normal periods
    if quarter <= 4:
        return ((4 - quarter) * 720) + quarter_time

    # overtime periods
    else:
        overtime_period = quarter - 4
        
        return ((overtime_period - 1) * 300) + quarter_time

In [ ]:
df["game_seconds_remaining"] = 0.0

regulation_mask = df["period"] <= 4

df.loc[regulation_mask, "game_seconds_remaining"] = (
    (4 - df.loc[regulation_mask, "period"]) * 720
    + df.loc[regulation_mask, "quarter_seconds_remaining"]
)

In [ ]:
overtime_mask = df["period"] > 4

df.loc[overtime_mask, "game_seconds_remaining"] = (
    (df.loc[overtime_mask, "period"] - 5) * 300
    + df.loc[overtime_mask, "quarter_seconds_remaining"]
)

In [ ]:
df[["period","clock","game_seconds_remaining"]].head(20)

,period,clock,game_seconds_remaining
0,1,PT00M00.00S,2880.0
1,1,PT00M00.10S,2879.9
2,1,PT00M00.20S,2879.8
3,1,PT00M23.30S,2856.7
4,1,PT00M26.30S,2853.7
5,1,PT00M26.30S,2853.7
6,1,PT00M33.10S,2846.9
7,1,PT00M33.10S,2846.9
8,1,PT00M33.10S,2846.9
9,1,PT00M34.40S,2845.6


In [ ]:
df["score_diff_change"] = df.groupby("gameid")["score_diff"].diff().fillna(0)

In [ ]:
df["momentum"] = df.groupby("gameid")["score_diff_change"].transform(
    lambda x: x.ewm(span=15, adjust=False).mean()
)

In [ ]:
final_scores = df.groupby("gameid").tail(1)

game_results = final_scores[["gameid", "h_pts", "a_pts"]].copy()
game_results["home_win"] = (game_results["h_pts"] > game_results["a_pts"]).astype(int)

In [ ]:
df = df.merge(game_results[["gameid", "home_win"]], on="gameid", how="left")

In [ ]:
df[[
    "period",
    "clock",
    "quarter_seconds_elapsed",
    "quarter_seconds_remaining",
    "game_seconds_remaining"
]].head(20)

,period,clock,quarter_seconds_elapsed,quarter_seconds_remaining,game_seconds_remaining
0,1,PT00M00.00S,0.0,720.0,2880.0
1,1,PT00M00.10S,0.1,719.9,2879.9
2,1,PT00M00.20S,0.2,719.8,2879.8
3,1,PT00M23.30S,23.3,696.7,2856.7
4,1,PT00M26.30S,26.3,693.7,2853.7
5,1,PT00M26.30S,26.3,693.7,2853.7
6,1,PT00M33.10S,33.1,686.9,2846.9
7,1,PT00M33.10S,33.1,686.9,2846.9
8,1,PT00M33.10S,33.1,686.9,2846.9
9,1,PT00M34.40S,34.4,685.6,2845.6


In [ ]:
ml_df = df[[
    "gameid",
    "season",
    "period",
    "game_seconds_remaining",
    "score_diff",
    "score_diff_squared",
    "momentum",
    "home_win"
]].copy()

In [ ]:
ml_df = ml_df.dropna().reset_index(drop=True)

In [ ]:
print("Shape:", ml_df.shape)
ml_df["home_win"].value_counts(normalize=True)

Shape: (18246068, 7)


home_win
1    0.5263
0    0.4737
Name: proportion, dtype: float64

In [ ]:
ml_df.to_parquet("../data/processed/ml_dataset.parquet", index=False)

In [ ]:
ml_df["period"].value_counts(normalize=True)

period
4    0.257910
2    0.254489
3    0.241632
1    0.237807
5    0.007035
6    0.000962
7    0.000145
8    0.000021
Name: proportion, dtype: float64

In [ ]:
ml_df["game_seconds_remaining"].describe()

count    1.825573e+07
mean     1.440376e+03
std      8.289373e+02
min      0.000000e+00
25%      7.049000e+02
50%      1.436400e+03
75%      2.143900e+03
max      2.880000e+03
Name: game_seconds_remaining, dtype: float64

In [ ]:
df["gameid"].nunique()

37928